12/09/2026
First version

In [4]:
from io import TextIOWrapper
from pathlib import Path

import chess
import chess.engine
import chess.pgn
import numpy as np
import pandas as pd
import zstandard as zstd
from sklearn.tree import DecisionTreeClassifier


In [7]:
PGN_ZST_PATH = Path("DB/lichess_db_standard_rated_2013-01.pgn.zst")


def partidas_en_stream(ruta: Path):
    """Genera partidas PGN una a una desde un archivo .pgn.zst."""
    with ruta.open("rb") as archivo_comprimido:
        descompresor = zstd.ZstdDecompressor()
        with descompresor.stream_reader(archivo_comprimido) as flujo_binario:
            flujo_texto = TextIOWrapper(flujo_binario, encoding="utf-8")
            while partida := chess.pgn.read_game(flujo_texto):
                yield partida


def partida_en_indice(ruta: Path, indice: int):
    """Devuelve la partida con índice cero-based sin cargar todas las partidas."""
    if indice < 0:
        raise ValueError("El índice debe ser mayor o igual que cero")

    for indice_actual, partida in enumerate(partidas_en_stream(ruta)):
        if indice_actual == indice:
            return partida

    raise IndexError(f"No existe una partida con índice {indice}")


INDICE_PARTIDA = 0
partida_seleccionada = partida_en_indice(PGN_ZST_PATH, INDICE_PARTIDA)
resultado_final = partida_seleccionada.headers.get("Result", "*")
tablero = partida_seleccionada.board()
movimientos = []

for movimiento in partida_seleccionada.mainline_moves():
    movimientos.append(tablero.san(movimiento))
    tablero.push(movimiento)

print("Índice:", INDICE_PARTIDA)
print("Resultado final:", resultado_final)
print("Movimientos:", " ".join(movimientos))


Índice: 0
Resultado final: 1-0
Movimientos: e4 e6 d4 b6 a3 Bb7 Nc3 Nh6 Bxh6 gxh6 Be2 Qg5 Bg4 h5 Nf3 Qg6 Nh4 Qg5 Bxh5 Qxh4 Qf3 Kd8 Qxf7 Nc6 Qe8#
